In [1]:
pip install confluent-kafka faker


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
import random
import time
import logging
from datetime import datetime, timezone
from faker import Faker
from confluent_kafka import Producer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

fake = Faker()

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
KAFKA_BROKERS    = os.environ.get("KAFKA_BROKERS",    "pkc-n98pk.us-west-2.aws.confluent.cloud:9092")
KAFKA_TOPIC      = os.environ.get("KAFKA_TOPIC",      "orders_topic")
KAFKA_API_KEY    = os.environ.get("KAFKA_API_KEY",    "GIITD77LNNPTLUPY")
KAFKA_API_SECRET = os.environ.get("KAFKA_API_SECRET", "cfltGr8vg7/QUyxTssw3J2azvM7LLVn7DuZ/lVr+XscdG5uY/X9mEp3g6v6fG80g")
RATE_PER_SEC     = float(os.environ.get("RATE_PER_SEC", "2.0"))   # msgs / second

conf = {
    "bootstrap.servers":  KAFKA_BROKERS,
    "security.protocol":  "SASL_SSL",
    "sasl.mechanism":     "PLAIN",
    "sasl.username":      KAFKA_API_KEY,
    "sasl.password":      KAFKA_API_SECRET,
    "client.id":          "python-faker-producer",
    "compression.type":   "snappy",
}

producer = Producer(conf)

PRODUCTS = [
    "Wireless Mouse", "Mechanical Keyboard", "USB-C Hub", "4K Monitor",
    "Noise Cancelling Headphones", "Webcam 1080p", "Laptop Stand",
    "SSD 1TB", "RAM 32GB", "Gaming Chair", "Smart Watch", "USB Microphone"
]

def delivery_report(err, msg):
    """Callback invoked on message delivery success or failure."""
    if err is not None:
        logger.error(f"Delivery failed: {err}")
    else:
        logger.debug(
            f"Sent to {msg.topic()} partition {msg.partition()} offset {msg.offset()}"
        )

def generate_order() -> dict:
    """
    Generate a single fake order.
    Schema matches the Glue Spark schema exactly:
      order_id:string, customer_id:string, product:string,
      quantity:int, price:double, order_timestamp:timestamp
    """
    return {
        "order_id":        fake.uuid4(),
        "customer_id":     fake.uuid4(),
        "product":         random.choice(PRODUCTS),
        "quantity":        random.randint(1, 5),
        "price":           round(random.uniform(9.99, 2499.99), 2),
        "order_timestamp": datetime.now(timezone.utc).isoformat()
    }

def main():
    logger.info(f"Starting producer | Topic: {KAFKA_TOPIC} | Rate: {RATE_PER_SEC} msg/s")
    msg_count = 0
    sleep_interval = 1.0 / RATE_PER_SEC

    try:
        while True:
            order = generate_order()
            print(order)
            # send the orders to orders_topic
            producer.produce(
                topic=KAFKA_TOPIC,
                key=order["order_id"],                       # good for partitioning
                value=json.dumps(order, default=str).encode("utf-8"),
                callback=delivery_report
            )

            # Trigger delivery callbacks without blocking
            producer.poll(0)

            msg_count += 1
            if msg_count % 100 == 0:
                logger.info(f"Produced {msg_count} messages...")

            time.sleep(sleep_interval)

    except KeyboardInterrupt:
        logger.info("Shutdown requested.")
    finally:
        logger.info("Flushing remaining messages...")
        producer.flush()
        logger.info(f"Producer stopped. Total produced: {msg_count}")

if __name__ == "__main__":
    main()

2026-07-16 07:29:23,186 [INFO] Starting producer | Topic: orders_topic | Rate: 2.0 msg/s


{'order_id': 'ade66f9f-be09-4a24-adea-bbe1a9231db9', 'customer_id': '38bd47e6-d818-4911-b0cb-4cb1445d893c', 'product': 'Smart Watch', 'quantity': 5, 'price': 2314.42, 'order_timestamp': '2026-07-16T14:29:23.187617+00:00'}
{'order_id': '5407d15b-2d29-4c45-ac34-9576ab09db55', 'customer_id': '2dce1a58-55f3-4032-8a6e-77f51437efb0', 'product': 'Smart Watch', 'quantity': 4, 'price': 2340.76, 'order_timestamp': '2026-07-16T14:29:23.692807+00:00'}


%6|1784212163.830|GETSUBSCRIPTIONS|python-faker-producer#producer-5| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to 8aPVx4XQTBSVLwdZzMJ06A


{'order_id': '4832f675-a204-4df6-9162-8b61dfb5497c', 'customer_id': 'bb1a299c-8466-4776-adff-e776db1171d4', 'product': 'Noise Cancelling Headphones', 'quantity': 1, 'price': 2476.59, 'order_timestamp': '2026-07-16T14:29:24.197293+00:00'}
{'order_id': 'd171b11b-65f6-4ab4-b11f-9bbf9781f007', 'customer_id': 'b4e413d4-0358-4448-9e0e-d12ac19a27a5', 'product': 'Smart Watch', 'quantity': 2, 'price': 2402.17, 'order_timestamp': '2026-07-16T14:29:24.698734+00:00'}
{'order_id': '177733d3-a730-4be8-ad22-1199bd03eded', 'customer_id': 'c4a72299-f47b-4f62-b0b3-de8d1a041c16', 'product': 'Smart Watch', 'quantity': 5, 'price': 1651.26, 'order_timestamp': '2026-07-16T14:29:25.204120+00:00'}
{'order_id': 'fb0094ac-d34e-4edd-bcb2-ce7dcf82dd49', 'customer_id': '5af42885-2deb-427f-9935-67cc636511a6', 'product': '4K Monitor', 'quantity': 1, 'price': 947.12, 'order_timestamp': '2026-07-16T14:29:25.708190+00:00'}
{'order_id': '8b69080e-16fb-48e5-a7f1-20b9f83d71be', 'customer_id': '4ab056b9-fac5-4e22-a2b9-097db

2026-07-16 07:30:13,089 [INFO] Produced 100 messages...


{'order_id': 'a6d1bf7f-8e91-4912-b840-377ce1942559', 'customer_id': '0f21559e-5c63-4aab-a62d-531a7e267a6c', 'product': 'Laptop Stand', 'quantity': 2, 'price': 1203.38, 'order_timestamp': '2026-07-16T14:30:13.089054+00:00'}
{'order_id': 'e69ed3af-c7c5-40f6-a2e6-2d528873e113', 'customer_id': '197af922-e591-4670-8fe1-5d5057c73dfe', 'product': 'Wireless Mouse', 'quantity': 5, 'price': 2359.72, 'order_timestamp': '2026-07-16T14:30:13.593988+00:00'}
{'order_id': 'dcfeb85a-354c-457c-ad70-6ff1aa177a48', 'customer_id': 'e1d42429-5900-4d66-b9af-a0d7a2ef6089', 'product': 'RAM 32GB', 'quantity': 5, 'price': 1448.8, 'order_timestamp': '2026-07-16T14:30:14.097940+00:00'}
{'order_id': '5e2fa2db-1463-4646-a0be-8c6836889210', 'customer_id': 'c3c54cf9-96c1-486a-9368-ebd9c9d40320', 'product': 'Gaming Chair', 'quantity': 4, 'price': 1666.47, 'order_timestamp': '2026-07-16T14:30:14.603329+00:00'}
{'order_id': '25cfa009-442f-4e4c-a6b0-ed4d0ee4f3c0', 'customer_id': '8cc75bfd-de8a-438a-a22b-6da4c1de98dd', 'pr

2026-07-16 07:30:28,853 [INFO] Shutdown requested.
2026-07-16 07:30:28,853 [INFO] Flushing remaining messages...
2026-07-16 07:30:28,854 [INFO] Producer stopped. Total produced: 131


{'order_id': '714425fe-2a55-4a66-bcff-d83f163e60ce', 'customer_id': 'c91335c2-c9aa-432f-81ad-ac1a65f5a556', 'product': 'Laptop Stand', 'quantity': 3, 'price': 1794.16, 'order_timestamp': '2026-07-16T14:30:28.697909+00:00'}
